In [ ]:
import datetime
import os
from functools import partial
from itertools import product
from pathlib import Path
from string import ascii_lowercase

os.environ["POLARS_FORCE_NEW_STREAMING"]="1"

import colormaps
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import xarray as xr
from matplotlib.colors import (
    BoundaryNorm,
    LinearSegmentedColormap,
    Normalize,
    hsv_to_rgb,
    rgb_to_hsv,
)
from matplotlib.dates import DateFormatter
from matplotlib.ticker import MaxNLocator
from tqdm import tqdm, trange

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

import os
from functools import partial
from itertools import product
from pathlib import Path

import dask.array as darr
import numpy as np
import polars as pl
import xarray as xr
from scipy.constants import g
from scipy.signal.windows import lanczos
from tqdm import tqdm, trange
from zarr.codecs.numcodecs import Blosc

from jetutils.anyspell import (
    extend_spells,
    get_spells,
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
)
from jetutils.data import (
    compute_anomalies_pl,
    extract,
    open_da,
    smooth,
    standardize,
    standardize_polars_dtypes,
    compute_all_smoothed_anomalies,
)
from jetutils.definitions import (
    C_P_AIR,
    DATADIR,
    DEFAULT_VARNAME,
    DUNCANS_REGIONS_NAMES,
    FACTORS,
    FACTORS_UNITS,
    FIGURES,
    KAPPA,
    MONTH_NAMES,
    N_WORKERS,
    OMEGA,
    PRETTIER_VARNAME,
    RADIUS,
    RESULTS,
    UNITS,
    YEARS,
    compute,
    degcos,
    degsin,
    get_index_columns,
    get_region,
    polars_to_xarray,
    squarify,
    to_expr,
    xarray_to_polars,
)
from jetutils.derived_quantities import (
    compute_2d_div,
    compute_absolute_vorticity,
    compute_norm_derivative,
    convolve_dask,
)
from jetutils.geospatial import (
    central_diff,
    compute_relative_anom,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    create_all_relative_plots,
    create_bias_correction,
    create_jet_relative_dataset,
    detect_contours,
    detect_contours_lonlat,
    detect_overturnings,
    detect_streamers,
    event_props,
    gather_normal_da_jets,
    haversine_from_dl,
    interp_jets_to_zero_one,
    to_xarray_sjoin,
    common_relative_plot,
)
from jetutils.jet_finding import (
    DataHandler,
    JetFindingExperiment,
    add_feature_for_cat,
    average_jet_categories,
    connected_from_cross,
    do_everything,
    extract_features,
    find_all_jets,
    gaussian_smooth_func,
    iterate_over_year_maybe_member,
    jet_position_as_da,
    slowness_expr,
    slowness_from_cross,
    spells_from_cross,
    spells_from_cross_catd_simple,
    to_one_large,
    track_jets,
)
from jetutils.plots import (
    COLORS,
    COLORS_EXT,
    STYLE_SHEET,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    gaussian_kde,
    plot_interp,
    plot_relative_time,
    props_vs_quantiles,
)
from jetutils.stats import (
    common_OR,
    create_bootstrapped_times,
    is_signif_OR,
    odds_ratio,
)

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

basepath = Path(f"{DATADIR}/ERA5/plev/high_wind/6H/results/9")


ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)

In [ ]:
jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
phat_jets = to_one_large(jets)
props = pl.read_parquet(basepath.joinpath("props.parquet"))
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))
pers = slowness_from_cross(cross).drop("is_polar")
props = props.join(pers, on=["time", "jet ID"])

over_europe = pl.col("lon") > -10
lat_over_europe = (pl.col("lat") * pl.col("s")).filter(over_europe).sum() / pl.col(
    "s"
).filter(over_europe).sum()
lat_over_europe = jets.group_by("time", "jet ID").agg(
    lat_over_europe.fill_nan(0).alias("lat_over_europe")
)
props = props.join(lat_over_europe, on=["time", "jet ID"])

props_catd = squarify(average_jet_categories(props), ["time", "jet"])
props_catd = props_catd.join(
    props_catd.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
pers = props_catd.rolling("time", period="5d", group_by="jet").agg(
    pers=pl.col("slowness").sum()
)
props_catd = props_catd.join(pers, on=("time", "jet"))
props_catd_summer = summer_filter.join(props_catd, on="time")

dist_threshold = 2.0e6
overlap_threshold = 0.6
dis_polar_thresh = 0.12
spells_list = spells_from_cross(
    jets,
    cross,
    None,
    dist_threshold,
    overlap_threshold,
    dis_polar_thresh,
    n_STJ=30,
    n_EDJ=30,
    season=summer,
    STJ_lat_threshold=30,
)
_, summary_comp = connected_from_cross(
    jets,
    cross,
    dist_threshold,
    overlap_threshold,
    dis_polar_thresh,
)
jet_column = (
    pl.when((pl.col("is_polar") > 0.5).mean().over("spell") > 0.5)
    .then(pl.lit("EDJ"))
    .otherwise(pl.lit("STJ"))
)
summary_comp = (
    summary_comp.filter(pl.col("len") > 1)
    .drop("s", "theta", "is_polar", "s_right", "theta_right", "is_polar_right")
    .join(props["time", "jet ID", "is_polar"], on=("time", "jet ID"))
    .with_columns(
        jet=jet_column,
        slowness=slowness_expr(),
    )
    .with_columns(
        slowness_sum=pl.col("slowness").sum().over("spell"),
    )
    # .join(props, on=["time", "jet ID"], suffix="_fromprops")
    .sort("time", "jet ID")
)
summer_summary_comp = summary_comp.filter(
    pl.col("time")
    .is_in(pl.lit(summer.implode().first(), pl.List(pl.Datetime("ms"))))
    .over("spell")
)

for jet_name, spells in spells_list.items():
    print(jet_name, spells["spell"].n_unique())
    print(jet_name, spells["len"].min() / 4)

daily_spells_list = {
    a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()
}

# fake eke

In [ ]:
from jetutils.geospatial import nearest_mapping
bias_correction = pl.read_parquet(basepath.joinpath("bias_correct.parquet"))
tf = pl.col("time").dt.year() == 1959
jets_ = jets.filter(tf)
bias_correction_ = bias_correction.filter(tf)
ds = xr.open_dataset(f"{DATADIR}/ERA5/plev/uv/6H/full.zarr", consolidated=False)
ds = ds.sel(time=ds.time.dt.year==1959, lev=250)
das = [ds["u"], ds["v"], np.hypot(ds["u"], ds["v"]).rename("s")]
das = [da.coarsen({"lon": 3, "lat": 3}, boundary="pad").mean().load() for da in das]
half_length = 2e6
dn = 1e5
in_meters = True
index_columns = get_index_columns(
    jets,
    (
        "member",
        "time",
        "cluster",
        "spell",
        "relative_index",
        "relative_time",
        "sample_index",
        "inside_index",
    ),
)
which_jet = "jet" if "jet" in jets.columns else "jet ID"
extra_n = bias_correction_["n"].abs().max()
extra_n = (extra_n // dn) * dn
jets_with_interp = gather_normal_da_jets(
    jets_,
    *das,
    half_length=half_length + extra_n,
    dn=dn,
    in_meters=in_meters,
)
mapping = nearest_mapping(bias_correction_, jets_with_interp, "n")
bias_correction_ = (
    bias_correction_.join(mapping, on="n").drop("n").rename({"n_": "n"})
)
jets_with_interp = (
    jets_with_interp.join(
        bias_correction_, on=[*index_columns, which_jet, "index"]
    )
    .with_columns(n=pl.col("n") - pl.col("n_right"))
    .filter(pl.col("n").abs() <= half_length)
    .with_columns(side=pl.col("n").sign().cast(pl.Int32()))
    .drop("n_right", "lon_right", "lat_right", "angle_right")
)
jets_with_interp = (
    jets_with_interp
    .filter(pl.col("n").abs() <= half_length)
    .with_columns(pl.col("angle") - np.pi/2)
)
jets_with_interp = jets_with_interp.join(jets_with_interp.filter(pl.col("n") == 0).select("time", "jet ID", "index", "u_interp", "v_interp", "s_interp").rename({f"{var}_interp": f"{var}_centre" for var in ["u", "v", "s"]}), on=["time", "jet ID", "index"])
jets_with_interp.write_parquet(f"{RESULTS}/playtest.parquet")

In [ ]:
period = 11
offset = int(np.ceil(period / 2))
bees = (
    jets_with_interp
    .select("time", "jet ID", "index", "n", s_ratio=pl.col("s_interp") / pl.col("s_centre"))
    .filter(pl.col("n") != 0, pl.col("n").abs() < 7e5)
    .filter(pl.col("s_ratio") < 1)
    .with_columns(side=pl.col("n").sign(), b=pl.col("n").abs() / pl.col("s_ratio").log().abs().sqrt())
    .group_by("time", "jet ID", "index", "side", maintain_order=True)
    .agg(pl.col("b").mean())
    # .rolling("index", group_by=["time", "jet ID", "side"], period=f"{period}i", offset=f"-{offset}i")
    # .agg(pl.col("b").mean())
    .sort("time", "jet ID", "index", "side")
)
bees = (
    squarify(bees, [("time", "jet ID", "index"), "side"])
    .with_columns(pl.col("b").fill_nan(None).fill_null(strategy="mean").over(["time", "jet ID", "index"]))
)
gs_expr = pl.col("s_centre") * (-pl.col("n").pow(2) / pl.col("b").pow(2)).exp()
jets_with_interp = (
    jets_with_interp
    .with_columns(side=pl.col("n").sign().cast(pl.Float32()))
    .join(bees, on=["time", "jet ID", "index", "side"])
    .with_columns(s_gaussian=gs_expr)
)

In [ ]:
smoothed = []
for nabs, jwi in jets_with_interp.group_by(pl.col("n").abs()):
    nabs = nabs[0]
    period = int(nabs / 1e5)
    offset = int(np.ceil(period / 2))
    if nabs == 0:
        smoothed.append(jwi.select("time", "jet ID", "n", "index", "s_interp"))
        continue
    break
    jwi = (
        jwi
        .rolling("index", group_by=("time", "jet ID", "n"), period=f"{period}i", offset=f"-{offset}i")
        .agg(pl.col("s_interp").mean())
    )
    smoothed.append(jwi)
# smoothed = pl.concat(smoothed).sort("time", "jet ID", "index", "n")
# jets_with_interp = jets_with_interp.join(smoothed.rename({"s_interp": "s_smooth"}), on=["time", "jet ID", "index", "n"])

In [ ]:
jwi

In [ ]:
da_df = polars_to_xarray(jets_with_interp["time", "jet ID", "index", "n", "s_interp", "s_smooth", "s_gaussian"], ["time", "jet ID", "n", "index"])
n_t, n_i = 200, 19
da_df["s_interp"][n_t,0,:,n_i].plot()
da_df["s_smooth"][n_t,0,:,n_i].plot()
da_df["s_gaussian"][n_t,0,:,n_i].plot()

In [ ]:
da_df = polars_to_xarray(jets_with_interp["time", "jet ID", "normallon_rounded", "normallat_rounded", "is_polar", "u_interp", "v_interp"].unique(["time", "jet ID", "normallon_rounded", "normallat_rounded"]), ["time", "jet ID", "normallat_rounded", "normallon_rounded"])

In [ ]:
da_df["u_interp"][1250, 1].plot()

In [ ]:
ds["u"][1250].plot()

# relative_clims

In [ ]:
variables = [
    "theta",
    "theta:grad",
    "t2m",
    "tp",
    "eady", 
    "eady_s",
]
for level in [200, 225, 250, 300, 350, 850]:
    variables.append(f"t{level}")
    variables.append(f"t{level}:grad")
n_days = 5
for level in [200, 225, 250, 850]:
    for var in ["EKE", "hor", "vert"]:
        variables.append(f"{var}{level}_{n_days}days")
for var in ["APVO", "CPVO"]:
    for level in [320, 330, 340, 350, "any"]:
        variables.append(f"{var}{level}")
for level in [320, 330, 340, 350]:
    for var, days in product(["EKE", "EMFconv"], [10, 20, 30]):
        variables.append(f"{var}{level}_{days}days")
    variables.append(f"PV{level}")
    variables.append(f"PV{level}:grad")

for var in tqdm(variables):    
    common_relative_plot(var, basepath, "STJ", summer)

# new uv data

What do I want ?

- full EP Flux (vert and hor like Henrik jets)
- jet-centred T at all altitudes, mostly for gradients
- vertical derivatives / eady growth rate
- jet centred omega around STJ especially
- box average omega for convection?

In [ ]:
from scipy.constants import g
from scipy.signal.windows import lanczos
from zarr.codecs.numcodecs import Blosc

from jetutils.definitions import C_P_AIR, KAPPA, OMEGA, compute, degcos, degsin
from jetutils.derived_quantities import compute_2d_div, convolve_dask

Basepath = Path(DATADIR, "ERA5/plev/uv/6H")
ozarr = Basepath.joinpath("full.zarr")
n_days = 10
half_len = n_days * 2
ds = xr.open_dataset(ozarr, consolidated=False).chunk({"time": 200, "lev": -1, "lat": -1, "lon": -1})
ds["theta"] = ds["t"] * (1000 / ds["lev"]) ** KAPPA
ds["theta"] = ds["theta"]
ds = ds.drop_vars(["t", "omega"])
N = np.sqrt(g / ds["theta"] * ds["theta"].differentiate("lev"))
N = N.astype(np.float32)
f = (2 * OMEGA * degsin(ds.lat)).astype(np.float32)
eady1 = 0.3098 * g * np.abs(f) * abs(ds["u"].differentiate("lev")) / N
eady2 = 0.3098 * g * np.abs(f) * abs(ds["v"].differentiate("lev")) / N

ds_eady = xr.Dataset({"eady1": eady1, "eady2": eady2})
ds_eady = ds_eady.sel(lev=[850, 250, 225, 200])

encoding = {}
compressor = Blosc(cname="zstd", clevel=3, shuffle=2)
for data_var in ds_eady.data_vars:
   encoding[data_var] = {"compressors": compressor}

res = ds_eady.to_zarr(Basepath.joinpath("results/eady_growth.zarr"), compute=False, consolidated=False, encoding=encoding)
compute(res, progress_flag=True)

In [ ]:
from scipy.signal.windows import lanczos
from zarr.codecs.numcodecs import Blosc

from jetutils.definitions import C_P_AIR, KAPPA, OMEGA, compute, degcos, degsin
from jetutils.derived_quantities import compute_2d_div, convolve_dask

Basepath = Path(DATADIR, "ERA5/plev/uv/6H")
ozarr = Basepath.joinpath("full.zarr")
n_days = 10
half_len = n_days * 2
ds = xr.open_dataset(ozarr, consolidated=False).chunk({"time": 1000, "lev": 1, "lat": -1, "lon": -1})
ds["theta"] = ds["t"] * (1000 / ds["lev"]) ** KAPPA

l_win = lanczos(2 * half_len + 1)[:, None, None, None]
dims = ds.dims
for var in ["u", "v", "theta", "omega"]:
    ds[f"{var}bar"] = (
        dims,
        (
            convolve_dask(ds[var].data, l_win)[half_len:-half_len] / l_win.sum()
        ).astype(np.float32),
    )
    ds[f"{var}p"] = ds[var] - ds[f"{var}bar"]
    del ds[f"{var}bar"]
    del ds[var]
encoding = {}
compressor = Blosc(cname="zstd", clevel=3, shuffle=2)
for data_var in ds.data_vars:
   encoding[data_var] = {"compressors": compressor}

res = ds.to_zarr(Basepath.joinpath("results/eddy_stuff_10days.zarr"), compute=False, consolidated=False, encoding=encoding)
compute(res, progress_flag=True)

In [ ]:
Basepath = Path(DATADIR, "ERA5/plev/uv/6H")
izarr1 = Basepath.joinpath("full.zarr")
izarr2 = Basepath.joinpath("results/eddy_stuff_10days.zarr")
ds = xr.open_dataset(izarr2, consolidated=False).chunk({"time": 100, "lev": -1, "lat": -1, "lon": -1})
theta = xr.open_dataset(izarr1, consolidated=False).chunk({"time": 100, "lev": -1, "lat": -1, "lon": -1})["t"]
theta = theta * (1000 / theta["lev"]) ** KAPPA
ds["dthetadp"] = theta.differentiate("lev")
ds["EKE"] = (0.5 * (ds["up"] ** 2 + ds["vp"] ** 2)).astype(np.float32)
gamma = (-KAPPA / ds.lev * (1000 / ds.lev) ** KAPPA * ds["dthetadp"].mean(["time", "lon", "lat"])).astype(np.float32)
EAPE = (C_P_AIR * 0.5 * (ds.lev * 1e-3) ** (2 * KAPPA) * gamma * ds["thetap"] ** 2).astype(np.float32)
S = (ds["EKE"] - 0.5 * EAPE).astype(np.float32)
f = (2 * OMEGA * degsin(ds.lat)).astype(np.float32)

## Base 2 * 3
ds["F11"] = S - ds["up"] ** 2
ds["F12"] = - ds["up"] * ds["vp"]
ds["F13"] = ds["vp"] * ds["thetap"] * f / ds["dthetadp"]
ds["F22"] = S - ds["vp"] ** 2
ds["F23"] = ds["up"] * ds["thetap"] * f / ds["dthetadp"]

## Additional from original EP:
ds["F13_extra"] = - ds["up"] * ds["omegap"]
ds["F23_extra"] = - ds["vp"] * ds["omegap"]

for g in ds.data_vars:
    if g[0] == "F":
        ds[g] = ds[g] * RADIUS * degcos(ds.lat)

ds = ds.unify_chunks()
    
ds["hor1"] = ds.map_blocks(compute_2d_div, ["F11", "F12"], template=ds["F11"])
ds["hor2"] = ds.map_blocks(compute_2d_div, ["F12", "F22"], template=ds["F11"])

ds["vert1"] = (ds["F13"] + ds["F13_extra"]).differentiate("lev")
ds["vert2"] = (ds["F23"] + ds["F23_extra"]).differentiate("lev")

ds = ds.drop_vars(["up", "vp", "thetap", "omegap", "dthetadp", *[g for g in ds.data_vars if g[0] == "F"]])
ds = ds.sel(lev=[850, 250, 225, 200])

encoding = {}
compressor = Blosc(cname="zstd", clevel=3, shuffle=2)
for data_var in ds.data_vars:
   encoding[data_var] = {"compressors": compressor}

res = ds.to_zarr(Basepath.joinpath("results/eddy_forcing_10days.zarr"), compute=False, consolidated=False, encoding=encoding)
compute(res, progress_flag=True)

# grams weather regimes

In [ ]:
dh = DataHandler.from_specs(
    "ERA5",
    "plev",
    "z",
    "6H",
    "all",
    [6, 7, 8, 9],
    -80,
    40,
    15,
    80,
    500,
    "dayofyear",
    {"dayofyear": ("win", 15)},
    None,
)

exp_z = Experiment(dh)
centers_z, labels_z = exp_z.do_kmeans(4, 30, weigh_grams=False)

coslat = np.cos(np.deg2rad(centers_z.lat))
Pwr = (
    (dh.da * centers_z * coslat).sum(["lon", "lat"])
    / coslat.sum()
    / centers_z.lon.shape[0]
)
Iwr = (Pwr - Pwr.mean("time")) / Pwr.std("time", ddof=0)
Iwr = compute(Iwr)
Iwr = xarray_to_polars(Iwr.rename("Iwr")).drop("ratio", "label")

# Iwr = Iwr.with_columns(year=pl.col("time").dt.year())
sigma_wr = Iwr["Iwr"].std()
winner = Iwr.group_by("time", maintain_order=True).agg(
    pl.col("Iwr").arg_max().alias("winner"),
    pl.col("Iwr").max(),
    pl.col("time").dt.year().alias("year").first(),
)
winner = winner.with_columns(
    winner=pl.when(pl.col("Iwr") > sigma_wr)
    .then(1 + pl.col("winner"))
    .otherwise(pl.lit(0))
)
start_of_year = (
    winner.group_by("year", maintain_order=True)
    .len()
    .with_columns(start_of_year=pl.col("len").cum_sum() - pl.col("len").get(0))
    .drop("len")
)
to_zero = (
    winner.group_by("year", maintain_order=True)
    .agg(pl.col("winner").rle().alias("rle"))
    .explode("rle")
    .unnest("rle")
    .group_by("year", maintain_order=True)
    .agg(
        len=pl.col("len"),
        start=pl.lit(0).append(
            pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1)
        ),
        value=pl.col("value"),
    )
    .explode(["len", "start", "value"])
    .join(start_of_year, on="year")
    .with_columns(start=pl.col("start") + pl.col("start_of_year"))
    .drop("start_of_year")
    .filter(pl.col("len") < 20, pl.col("value") > 0)
    .drop("value")
    .with_columns(index=pl.int_ranges(pl.col("start"), pl.col("start") + pl.col("len")))
    .drop("len", "start")
    .explode("index")
)[:, "index"]
winner[to_zero, "winner"] = 0

mask_updated = labels_to_mask(winner[:, "winner"].to_numpy())

In [ ]:
["Greenland Blocking", "Zonal regime", "EuBL/AR", "ScBl"]

In [ ]:
clu = Clusterplot(2, 2, get_region(exp_z.da))
clu.add_contourf(
    [dh.da[winner["winner"] == i].mean("time") / 9.8 for i in range(1, 5)],
    cmap=colormaps.BlWhRe,
    levels=[-100, -60, -20, 20, 60, 100],
    titles=[str(i) for i in range(1, 5)],
)
a = 1

In [ ]:
winner.write_parquet(dh.path.joinpath("grams_wr.parquet"))

# Real space composites

## data over europe

In [ ]:
compute_all_smoothed_anomalies("ERA5", "surf", "tp", "6H", "dayofyear", {"dayofyear": ("win", 15)})

## data over Natl

In [ ]:
# args1 = ["all", [6, 7, 8, 9], *get_region(ds)]
args2 = ["dayofyear", {"dayofyear": ("win", 15)}]

# da_tp = DataHandler.from_specs(
#     "ERA5", "surf", "tp", "6H", "all", [6, 7, 8, 9], *(-80, 40, 0, 90)
# ).da
# da_tp = compute(da_tp, progress_flag=True)
da_tp_anom = open_da(
    "ERA5", "surf", "tp", "6H", "all", [6, 7, 8, 9], *(-180, 180, -15, 70), "all",
    *args2,
)
# da_tp_anom = compute(da_tp_anom, progress_flag=True)
# da_theta2pvu = DataHandler.from_specs(
#     "ERA5", "surf", ("alot2pvu", "theta"), "6H", *args1
# ).da
# da_theta2pvu = compute(da_theta2pvu, progress_flag=True)
# da_T_anom = DataHandler.from_specs(
#     "ERA5",
#     "surf",
#     "t2m",
#     "6H",
#     "all",
#     [6, 7, 8, 9],
#     *(-80, 40, 0, 90),
#     "all",
#     *args2,
# ).da
# da_T_anom = compute(da_T_anom, progress_flag=True)
# da_z500_anom = DataHandler.from_specs(
#     "ERA5",
#     "plev",
#     "z",
#     "6H",
#     *args1,
#     500,
#     *args2,
# ).da
# da_z500_anom = compute(da_z500_anom, progress_flag=False).rename("z500") / 9.8
# varnames_rwb = ["APVO_new", "CPVO_new"]
# das_rwb = {}
# for var in varnames_rwb:
#     das_rwb[var] = compute(
#         DataHandler.from_specs(
#             "ERA5", "thetalev", var, "6H", *args1, reduce_da=False
#         ).da,
#         progress_flag=True,
#     )
# # ds_ = compute(ds.isel(time=np.isin(ds.time.dt.month, [6, 7, 8, 9])).chunk("auto"), progress_flag=True)

In [ ]:
ugh = da_tp_anom.coarsen({"lon": 4, "lat": 4}, boundary="pad", coord_func=np.nanmin).median().to_netcdf(f"{RESULTS}/tp_1deg.nc", compute=False)

In [ ]:
compute(ugh, progress_flag=True)

## create timeseries

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patches as mpatches
import matplotlib.path as mpath
import matplotlib.pyplot as plt
import shapely

from jetutils.data import extract, open_da

args2 = ["dayofyear", {"dayofyear": ("win", 15)}]

da_tp_anom = open_da(
    "ERA5", "surf", "tp", "6H", "all", None, *(-80, 40, 0, 90), "all",
    *args2,
)
da_tp_anom = compute(da_tp_anom, progress_flag=True)
da_T_anom = open_da(
    "ERA5",
    "surf",
    "t2m",
    "6H",
    "all",
    None,
    *(-80, 40, 0, 90),
    "all",
    *args2,
)
da_T_anom = compute(da_T_anom, progress_flag=True)

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.LambertConformal(-20)})
gl = ax.gridlines(
    draw_labels=False, linewidth=0.5, alpha=0.4, color="k", linestyle="--"
)
region = (-82, 42, -2, 80)
ax.set_boundary(
    mpath.Path(
        [
            [region[0], region[2]],
            [region[1], region[2]],
            [region[1], region[3]],
            [region[0], region[3]],
        ],
        codes=[1, 2, 2, 5],
    ).interpolated(20),
    transform=ccrs.PlateCarree(),
)
ax.add_feature(cfeature.COASTLINE)
ax.set_extent(region)

boxes_precip = {
    "tropical": (-80, 40, 0, 15),
    "gulfstream": (-80, -50, 20, 45),
    "mid_atl": (-60, -20, 35, 55),
}
boxes_temp = {
    "tropical": (-80, 40, 0, 15),
    "west": (-80, -40, 20, 60),
    "east": (-40, -0, 20, 60),
}

ax.add_feature(cfeature.COASTLINE)

for name, box in boxes_precip.items():
    ax.add_geometries(
        shapely.box(*np.asarray(box)[[0, 2, 1, 3]]).segmentize(10),
        facecolor="none",
        edgecolor="dodgerblue",
        linewidth=2,
        crs=ccrs.PlateCarree(),
    )
    huh = extract(da_tp_anom, "all", None, *box).mean(["lon", "lat"])
    opath = Path(DATADIR, "ERA5/timeseries/", f"{huh.name}_{name}.parquet")
    huh = xarray_to_polars(huh)
    huh.write_parquet(opath)
for name, box in boxes_temp.items():
    ax.add_geometries(
        shapely.box(*np.asarray(box)[[0, 2, 1, 3]]).segmentize(10),
        facecolor="none",
        edgecolor="red",
        ls="dashed",
        linewidth=2,
        crs=ccrs.PlateCarree(),
    )
    huh = extract(da_T_anom, "all", None, *box).mean(["lon", "lat"])
    opath = Path(DATADIR, "ERA5/timeseries/", f"{huh.name}_{name}.parquet")
    huh = xarray_to_polars(huh)
    huh.write_parquet(opath)
fig.savefig(f"{FIGURES}/jet_persistence/realspace_boxes_demo.pdf")

In [ ]:
# region_T_ts.write_parquet("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/surf/t2m/dailymean/dayofyear_doywin15/results/1/regionalized.parquet")

In [ ]:
# region_tp_ts.write_parquet("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/surf/tp/dailysum/dayofyear_doywin15/results/1/regionalized.parquet")

## Atlantic composites

In [ ]:
from jetutils.stats import field_significance_for_spells

to_plot = {}
pvals = {}
fdr_signif = {}

for da, jet in product([da_T_anom, da_tp_anom, da_z500_anom], ["STJ", "EDJ"]):
    key = f"{da.name}_{jet}"
    to_plot[key], pvals[key], fdr_signif[key] = field_significance_for_spells(da, spells_list[jet], summer, n_bootstraps=50, q=0.05)

In [ ]:
xr.Dataset(to_plot).to_netcdf(basepath.joinpath("NAtl.nc"))
xr.Dataset(pvals).to_netcdf(basepath.joinpath("NAtl_pvals.nc"))
xr.Dataset(fdr_signif).to_netcdf(basepath.joinpath("NAtl_fdr.nc"))

## Europe composites

#### just plot data

In [ ]:
from jetutils.stats import field_significance

reduction_function = np.nanmean
plt.rc("axes", titlesize=14)
da_contour = da_z500_anom.sel(time=da_z500_anom.time.dt.year >= 1959)
da_1 = da_T_anom_europe
da_2 = da_tp_anom_europe * 1000
days_around = 3
mask = np.zeros((len(da_contour.time), len(spells_list)), dtype=bool)
time_name = "time"
for j, jet in enumerate(spells_list):
    spells_from_jet = subset_around_onset(
        spells_list[jet], around_onset=datetime.timedelta(days=days_around)
    )
    mask[:, j] = np.isin(da_contour.time.values, spells_from_jet["time"])
    n_spells = spells_from_jet["spell"].n_unique()

to_plot = {}
signifs = {}
for da in [da_1, da_2]:
    to_plot_cf = []
    for mas in tqdm(mask.T, total=mask.shape[1]):
        if np.sum(mas) < 1:
            to_plot_cf.append(da[0].copy(data=np.zeros(da.shape[1:])))
            continue
        to_plot_cf.append(
            da.isel({time_name: mas}).reduce(reduction_function, dim=time_name)
        )
    to_plot[da.name] = xr.concat(to_plot_cf, dim="jet")

    to_test = []
    for mas in mask.T:
        if np.sum(mas) < 1:
            to_test.append(da[:1].copy(data=np.zeros((1, *da.shape[1:]))))
            continue
        to_test.append(da.isel(time=mas))
    significances = []
    da_ = da.values
    # da = np.sort(da, axis=0)
    for i in trange(mask.shape[1]):
        this_signif = field_significance(to_test[i], da_, 200, q=0.1)[1]
        this_signif = da[0].copy(data=this_signif).reset_coords("time", drop=True)
        significances.append(this_signif)
    signifs[da.name] = xr.concat(significances, dim="jet")

to_plot = xr.Dataset(to_plot)
signifs = xr.Dataset(signifs)

to_plot_c = []
for mas in tqdm(mask.T, total=mask.shape[1]):
    if np.sum(mas) < 1:
        to_plot_c.append(da_contour[0].copy(data=np.zeros(da.shape[1:])))
        continue
    to_plot_c.append(
        da_contour.isel({time_name: mas}).reduce(reduction_function, dim=time_name)
    )
to_plot_c = xr.concat(to_plot_c, dim="jet")

masked_da = []
time_name = "time"
reduction_function = np.nanmean
for mas in mask.T:
    masked_da.append(
        ds_.isel({time_name: mas}).reduce(reduction_function, dim=time_name)
    )
masked_da = xr.concat(masked_da, dim="relative_index")
find_jets_kwargs = dict(
    n_coarsen=1,
    base_s_thresh=20,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)
jets_on_mean = find_all_jets(masked_da, **find_jets_kwargs)

to_plot.to_netcdf(
    "/storage/homefs/hb22g102/jetutils/Results/figure_data_pers/contourf.nc"
)
signifs.to_netcdf(
    "/storage/homefs/hb22g102/jetutils/Results/figure_data_pers/signifs.nc"
)
to_plot_c.to_netcdf(
    "/storage/homefs/hb22g102/jetutils/Results/figure_data_pers/contour.nc"
)
masked_da.to_netcdf(
    "/storage/homefs/hb22g102/jetutils/Results/figure_data_pers/wind.nc"
)

## Individual spells

#### the case study plot

In [ ]:
from jetutils.data import coarsen_da, compute_anomalies_pl

n_stj = 23
n_edj = 9
figure = plt.figure(layout="constrained", figsize=(9, 5.6))
subfigs = figure.subfigures(1, 2, wspace=0.01)
fig1, fig2 = subfigs
nrow, ncol = 3, 2
n = nrow * ncol
plt.rc("axes", titlesize=14)
plt.rc("axes", titlepad=2)

rwb_color = {
    "S": "mediumseagreen",
    "T": "orangered",
}
rwb_hatching = {
    "A": r"xx",
    "C": r"++",
}
rwb_lw = 2

cbar_kwargs = {"location": "bottom", "shrink": 0.8, "fraction": 0.1, "pad": 0.03}
plot_kwargs_all = {
    "STJ": {
        "cmap": colormaps.cmp_b2r,
        "levels": MaxNLocator(7).tick_values(-10, 10).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
    "EDJ": {
        "cmap": colormaps.bilbao_r,
        "levels": MaxNLocator(8).tick_values(300, 390).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
}

for fig, key, da, which_spell in zip(
    subfigs, ["STJ", "EDJ"], [da_T_anom, da_theta2pvu], [n_stj, n_edj]
):
    is_edj = key == "EDJ"
    plot_kwargs = plot_kwargs_all[key]
    region = (-80, 40, 15, 80) if key == "EDJ" else (-80, 40, 0, 65)
    clu = Clusterplot(nrow, ncol, region, row_height=3.5, fig=fig)
    spell = spells_list[key].filter(pl.col("spell") == which_spell)
    spell = extend_spells(spell, time_before=datetime.timedelta(days=0))
    len_spell = spell["time"][-1] - spell["time"][0]

    times = (
        pl.linear_space(spell["time"][0], spell["time"][-1], n, eager=True)
        .dt.round("6h")
        .cast(pl.Datetime("ms"))
    )
    titles = times.dt.to_string("%Y-%m-%dT%H:00").to_list()
    titles = [
        f"{ascii_lowercase[n * is_edj + i]}) {title}" for i, title in enumerate(titles)
    ]
    to_plot = da.sel(time=times)
    clu.add_contourf(
        to_plot,
        titles=titles,
        cbar_label=PRETTIER_VARNAME.get(da.name, da.name),
        **plot_kwargs,
    )
    fig.suptitle(f"{key} episode, " + str(len_spell)[:6])

    for i, t in enumerate(times):
        if i == 0 and t < spell["time"][0]:
            t_ = spell["time"][0]
        else:
            t_ = t
        jets_ = jets.filter(pl.col("time") == t_)
        for _, jet in jets_.group_by("jet ID"):
            lo, la, is_p = jet[["lon", "lat", "is_polar"]]
            this_jet_is_p = len(is_p) > 0 and is_p.mean() > 0.4
            color = COLORS[1] if this_jet_is_p else COLORS[2]
            lw = 2 + 1 * (is_edj == this_jet_is_p)
            clu.axes[i].plot(lo - clu.central_longitude, la, color=color, lw=lw)
    # if key == "STJ":
    #     to_plot = da_z500_anom.sel(time=times)
    #     clu.add_contour(to_plot, levels=[-200, -100, 100, 200], colors="black", linewidths=2.)
    # if key == "STJ":
    #     to_plot = coarsen_da(da_tp.sel(time=times), 5)
    #     clu.add_contour(to_plot < 1e-5, levels=[0.5], colors=["green"], linewidths=3)
    #     break
    to_plot = coarsen_da(da_tp.sel(time=times), 3)
    clu.add_contour(to_plot * 1000, levels=[25], colors=["deepskyblue"], linewidths=1.4)
    if key == "EDJ":
        for (altitude, color), (orientation, hatch) in product(
            rwb_color.items(), rwb_hatching.items()
        ):
            to_plot = (
                das_rwb[f"{altitude}{orientation}PVS_new"].sel(time=times).any("lev")
            )
            for ax, ouais in zip(clu.axes, to_plot):
                cs = ax.pcolor(
                    to_plot.lon.values - clu.central_longitude,
                    to_plot.lat.values,
                    ouais.where(ouais),
                    hatch=hatch,
                    facecolor="none",
                    edgecolor=color,
                    hatch_linewidth=rwb_lw,
                    linewidth=0,
                    zorder=100,
                )

figure.savefig(f"{FIGURES}/jet_persistence/realspace_stuff/case_study_both.pdf")

In [ ]:
key = "EDJ"
which_spell = 1
spell = spells_list[key].filter(pl.col("spell") == which_spell).drop("jet ID")
spell = extend_spells(spell, time_before=datetime.timedelta(days=3))
spell = spell.filter((pl.col("relative_time") / datetime.timedelta(days=1)) % 3 == 0)
spell

## all of them

In [ ]:
from jetutils.data import coarsen_da, compute_anomalies_pl

n_t = 3
rwb_color = {
    "S": "mediumseagreen",
    "T": "orangered",
}
rwb_hatching = {
    "A": r"//",
    "C": r"\\",
}
rwb_lw = 2
cbar_kwargs = {"location": "bottom", "shrink": 0.7, "fraction": 0.1, "pad": 0.03}
plot_kwargs_all = {
    "STJ": {
        "da": da_T_anom,
        "cmap": colormaps.BlueWhiteOrangeRed,
        "levels": MaxNLocator(7).tick_values(-12, 12).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
    "EDJ": {
        "da": da_theta2pvu,
        "cmap": colormaps.bilbao_r,
        "levels": MaxNLocator(6).tick_values(300, 390).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
}
for spell_of in ["STJ", "EDJ"]:
    is_edj = spell_of == "EDJ"
    region = (-80, 40, 15, 80) if is_edj else (-80, 40, 0, 65)
    plot_kwargs = plot_kwargs_all[spell_of]
    spells = spells_list[spell_of].drop("jet ID")
    spells = extend_spells(spells, time_before=datetime.timedelta(days=3))
    spells = spells.filter((pl.col("relative_time") / datetime.timedelta(days=1)) % 3 == 0)
    da = plot_kwargs.pop("da")
    for n in range(30):
        clu = Clusterplot(n_t, 1, region, row_height=3.5)
        fig = clu.fig
        spell = spells.filter(pl.col("spell") == n)
        len_spell = spell["len"].first()
        times = spell["time"].unique().sort()[:n_t]
        titles = times.dt.to_string("%Y-%m-%dT%H:00").to_list()
        titles = [
            f"{ascii_lowercase[i]}) {title}" for i, title in enumerate(titles)
        ]
        to_plot = da.sel(time=times)
        clu.add_contourf(
            to_plot,
            titles=titles,
            cbar_label=PRETTIER_VARNAME.get(da.name, da.name),
            **plot_kwargs,
        )
        fig.suptitle(f"{spell_of} episode, " + str(len_spell)[:6])
        for i, t in enumerate(times):
            jets_ = jets.filter(pl.col("time") == t)
            for _, jet in jets_.group_by("jet ID"):
                lo, la, is_p = jet[["lon", "lat", "is_polar"]]
                this_jet_is_p = len(is_p) > 0 and (is_p < 0.5).mean() < 0.5
                this_jet_is_s = len(is_p) > 0 and (is_p < 0.5).mean() > 0.5
                color = COLORS[1] if this_jet_is_p else (COLORS[2] if this_jet_is_s else "gray")
                lw = 2 + 1 * (is_edj == this_jet_is_p)
                clu.axes[i].plot(lo - clu.central_longitude, la, color=color, lw=lw)
    
        to_plot = coarsen_da(da_tp_anom.sel(time=times), 3)
        color_tp = ["aquamarine"] if is_edj else ["cyan"]
        clu.add_contour(to_plot * 1000, levels=[15], colors=color_tp, linewidths=[1.4], linestyles=["solid"])
        if spell_of == "EDJ":
            for orientation, hatch in  rwb_hatching.items():
                to_plot = (
                    das_rwb[f"{orientation}PVO_new"].sel(time=times).any("lev")
                )
                for ax, ouais in zip(clu.axes, to_plot):
                    cs = ax.pcolor(
                        to_plot.lon.values - clu.central_longitude,
                        to_plot.lat.values,
                        ouais.where(ouais),
                        hatch=hatch,
                        facecolor="none",
                        edgecolor=rwb_color["S"],
                        hatch_linewidth=rwb_lw,
                        linewidth=0,
                        zorder=100,
                    )
        else:
            to_plot = da_z500_anom.sel(time=times)
            clu.add_contour(to_plot, levels=[-100, 100], colors="black", linewidths=2.)
        fig.savefig(f"{FIGURES}/jet_persistence/realspace_stuff/cs_{spell_of}_{n}.pdf")
        plt.close(fig)

    

In [ ]:
from scipy.stats import rankdata

da_tp_anom_ = da_tp_anom[:100, :20, :20]
da_tp_pvals = da_tp_anom_.copy(data=(rankdata(da_tp_anom_, axis=0) - 1) / len(da_tp_anom_.time))

In [ ]:
da_tp_pvals

In [ ]:
jets.filter(((pl.col("is_polar") < 0.5).n_unique() == 2).over("time", "jet ID")).group_by("time", "jet ID").agg(min=pl.col("is_polar").min(), max=pl.col("is_polar").max(), mean=pl.col("is_polar").mean(), prop_under=(pl.col("is_polar")<0.5).mean())

In [ ]:
to_save = {
    "t2m_anom": da_T_anom,
    "tp_anom": da_tp_anom,
    "tp": da_tp,
    "theta": da_theta2pvu,
    "z500_anom": da_z500_anom,
} | das_rwb
for name, da in to_save.items():
    for jet in ["STJ", "EDJ"]:
        spells = extend_spells(spells_list[jet], time_before=datetime.timedelta(days=4))
        spells = spells.select("spell", "relative_index", "time").unique(["spell", "relative_index"], maintain_order=True)
        to_save = []
        for _, spells_ in spells.group_by("spell", maintain_order=True):            
            times = polars_to_xarray(spells_, ["spell", "relative_index"])
            da_ = da.sel(time=times)
            to_save.append(da_)
        xr.concat(to_save, "spell").to_netcdf(basepath.joinpath(f"{name}_during_{jet}_spells.nc"))

In [ ]:
xr.open_dataarray(basepath.joinpath(f"{name}_during_{jet}_spells.nc"))

In [ ]:
times

In [ ]:
spells_list[jet]

In [ ]:
spells.unique(["spell", "relative_index"], maintain_order=True)

In [ ]:
polars_to_xarray(spells, ["spell", "relative_index"])

#